# 哔哩哔哩歌回路灯 (bili-song-light) 💡
### Bilibili Singing Timestamps Marker (Google Colab Version)

An audio digital signal processing (DSP) tool to automatically identify and mark singing segments in Bilibili recording streams or videos. Perfect for extracting singing timestamps and filtering out chat/talk segments.

一个基于数字信号处理（DSP）算法的 B 站录播/视频唱歌片段自动标记工具，全自动提取时间戳并过滤杂谈说话！

## 🛠️ Step 1: Install Dependencies & Speedup Tools / 安装依赖与下载加速器

Run the cell below to install the necessary libraries and the `aria2` multi-threaded downloader for much faster Bilibili streams downloading. FFmpeg is already pre-installed in the Google Colab environment.

In [ ]:
#@title Setup Environment
!apt-get update -qq && apt-get install -y -qq aria2
!pip install -q yt-dlp librosa numpy soundfile

## 📦 Step 2: Define Core Processing Heuristics / 定义核心处理逻辑

In [ ]:
import re
import shutil
import subprocess
import tempfile
from pathlib import Path
from dataclasses import dataclass

import numpy as np
import librosa
import soundfile as sf
from IPython.display import display, Markdown

@dataclass
class Segment:
    start: float
    end: float
    confidence: float
    loudness_lift: float = 0.0
    voice_presence: float = 0.0
    pitch_stability: float = 0.0

    @property
    def duration(self) -> float:
        return self.end - self.start

def download_audio(
    url: str,
    output_dir: Path,
    video_id: str,
    format_selector: str = "worstaudio",
    retries: int = 20
) -> Path:
    """Download audio using pure-Python yt-dlp API with custom user agent and aria2c speedup."""
    import yt_dlp
    import shutil
    target = output_dir / f"{video_id}.%(ext)s"
    
    ydl_opts = {
        'format': format_selector,
        'outtmpl': str(target),
        'noplaylist': True,
        'retries': retries,
        'fragment_retries': float('inf'),
        'quiet': True,
        'no_warnings': True,
        'http_headers': {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Referer': 'https://www.bilibili.com/',
            'Origin': 'https://www.bilibili.com',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7',
            'Accept-Language': 'zh-CN,zh;q=0.9,en;q=0.8,en-GB;q=0.7,en-US;q=0.6',
        }
    }
    
    # Auto-detect aria2c and apply multi-connection speedup
    if shutil.which("aria2c"):
        ydl_opts['external_downloader'] = 'aria2c'
        ydl_opts['external_downloader_args'] = {
            'default': ['-x', '16', '-s', '16', '-k', '1M', '--retry-wait=2', '--max-tries=0']
        }
        
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(url, download=True)
        filename = ydl.prepare_filename(info_dict)
        downloaded_path = Path(filename)
        if downloaded_path.exists():
            return downloaded_path
        else:
            raise FileNotFoundError(f"Download completed but file {downloaded_path} is missing.")

def convert_to_wav(source: Path, wav_path: Path) -> None:
    """Resample audio to 16kHz mono WAV using ffmpeg."""
    subprocess.run(
        [
            "ffmpeg",
            "-y",
            "-i",
            str(source),
            "-vn",
            "-ac",
            "1",
            "-ar",
            "16000",
            str(wav_path),
        ],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

def robust_scale(values: np.ndarray) -> np.ndarray:
    median = np.nanmedian(values)
    q1, q3 = np.nanpercentile(values, [25, 75])
    iqr = max(q3 - q1, 1e-6)
    return (values - median) / iqr

def moving_average(values: np.ndarray, width: int) -> np.ndarray:
    if width <= 1:
        return values
    kernel = np.ones(width) / width
    return np.convolve(values, kernel, mode="same")

def detect_segments(
    wav_path: Path,
    threshold: float,
    min_duration: float,
    merge_gap: float,
    min_loudness_lift: float = 0.35,
) -> list[Segment]:
    y, sr = librosa.load(wav_path, sr=16000, mono=True)
    hop_length = int(sr * 1.0)
    frame_length = int(sr * 2.0)

    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=frame_length, hop_length=hop_length)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr, hop_length=hop_length)[0]
    flatness = librosa.feature.spectral_flatness(y=y, hop_length=hop_length)[0]
    stft = np.abs(librosa.stft(y, n_fft=2048, hop_length=hop_length))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=2048)
    total_band = stft[(freqs >= 80) & (freqs <= 6000), :].sum(axis=0)
    vocal_band = stft[(freqs >= 180) & (freqs <= 3500), :].sum(axis=0)
    voice_presence = vocal_band / np.maximum(total_band, 1e-9)

    rms_db = librosa.amplitude_to_db(rms, ref=1.0)
    local_background = np.array(
        [
            np.nanpercentile(rms_db[max(0, i - 60) : min(len(rms_db), i + 61)], 25)
            for i in range(len(rms_db))
        ]
    )
    loudness_lift = np.clip((rms_db - local_background) / 12.0, 0.0, 1.5)

    pitches, magnitudes = librosa.piptrack(y=y, sr=sr, hop_length=hop_length, fmin=80, fmax=1000)
    voiced_pitch = []
    voiced_strength = []
    for frame in range(pitches.shape[1]):
        mag = magnitudes[:, frame]
        if mag.max() <= 0:
            voiced_pitch.append(np.nan)
            voiced_strength.append(0.0)
            continue
        idx = int(mag.argmax())
        voiced_pitch.append(float(pitches[idx, frame]))
        voiced_strength.append(float(mag[idx]))

    pitch = np.array(voiced_pitch)
    strength = np.array(voiced_strength)
    valid_pitch = np.isfinite(pitch) & (pitch > 0)

    pitch_stability = np.zeros_like(strength)
    for i in range(len(pitch)):
        lo = max(0, i - 4)
        hi = min(len(pitch), i + 5)
        window = pitch[lo:hi]
        window = window[np.isfinite(window) & (window > 0)]
        if len(window) >= 3:
            pitch_stability[i] = 1.0 / (1.0 + np.std(np.log2(window)))

    musical_energy = (
        0.25 * robust_scale(loudness_lift)
        + 0.25 * robust_scale(strength)
        + 0.20 * robust_scale(pitch_stability)
        + 0.15 * robust_scale(voice_presence)
        + 0.05 * robust_scale(bandwidth)
        - 0.10 * robust_scale(zcr)
        - 0.05 * robust_scale(flatness)
    )
    score = 1.0 / (1.0 + np.exp(-musical_energy))
    score = moving_average(score, width=9)
    score = np.where(valid_pitch, score, score * 0.5)
    score = np.where(loudness_lift >= 0.15, score, score * 0.65)

    active = score >= threshold
    raw: list[Segment] = []
    start = None
    for i, is_active in enumerate(active):
        if is_active and start is None:
            start = i
        if start is not None and (not is_active or i == len(active) - 1):
            end_idx = i if not is_active else i + 1
            conf = float(np.mean(score[start:end_idx]))
            raw.append(
                Segment(
                    float(start),
                    float(end_idx),
                    conf,
                    float(np.mean(loudness_lift[start:end_idx])),
                    float(np.mean(voice_presence[start:end_idx])),
                    float(np.mean(pitch_stability[start:end_idx])),
                )
            )
            start = None
            
    merged: list[Segment] = []
    for seg in raw:
        if not merged or seg.start - merged[-1].end > merge_gap:
            merged.append(seg)
        else:
            prev = merged[-1]
            total = prev.duration + seg.duration
            confidence = (prev.confidence * prev.duration + seg.confidence * seg.duration) / max(total, 1e-6)
            loudness = (prev.loudness_lift * prev.duration + seg.loudness_lift * seg.duration) / max(total, 1e-6)
            voice = (prev.voice_presence * prev.duration + seg.voice_presence * seg.duration) / max(total, 1e-6)
            pitch = (prev.pitch_stability * prev.duration + seg.pitch_stability * seg.duration) / max(total, 1e-6)
            merged[-1] = Segment(prev.start, seg.end, confidence, loudness, voice, pitch)
            
    return [
        seg
        for seg in merged
        if seg.duration >= min_duration and seg.loudness_lift >= min_loudness_lift
    ]

def fmt_time(seconds: float) -> str:
    seconds = int(round(seconds))
    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60
    if h:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

## 🎛️ Step 3: Configure Parameters & Run / 配置参数并运行

Fill in the Bilibili URL, adjust parameters using the sliders below, and run the cell to analyze. The cached download will be saved under the local folder `work/` in your Colab runtime.

In [ ]:
#@title Bilibili Song Marker Settings / B 站录播歌回路灯调参
url = "https://www.bilibili.com/video/BV1uUVf6QEMw" #@param {type:"string"}
threshold = 0.56 #@param {type:"slider", min:0.3, max:0.9, step:0.01}
min_duration = 25 #@param {type:"slider", min:5, max:180, step:1}
merge_gap = 30 #@param {type:"slider", min:5, max:120, step:1}
min_loudness_lift = 0.30 #@param {type:"slider", min:0.0, max:1.5, step:0.05}

import re
import tempfile
from pathlib import Path

match = re.search(r'(BV[a-zA-Z0-9]+|av[0-9]+)', url)
video_id = match.group(1) if match else "downloaded_audio"

work_dir = Path("work")
work_dir.mkdir(exist_ok=True)

print(f"Searching for video ID: {video_id}...")
existing_files = sorted(work_dir.glob(f"{video_id}.*"))
existing_files = [f for f in existing_files if f.suffix not in (".wav", ".part", ".ytdl")]

if existing_files:
    source = existing_files[0]
    print(f"-> Found cached recording: {source}, skipping download!")
else:
    print(f"-> Downloading Bilibili stream {video_id} (using worstaudio format with aria2c acceleration)... ")
    try:
        source = download_audio(url, work_dir, video_id, "worstaudio")
        print(f"-> Download complete: {source}")
    except Exception as e:
        print(f"-> Error downloading: {e}")
        source = None

if source and source.exists():
    with tempfile.TemporaryDirectory(prefix="song-marker-") as tmp:
        tmp_path = Path(tmp)
        wav_path = tmp_path / "audio.wav"
        
        print("-> Transcoding audio stream to 16kHz mono WAV...")
        convert_to_wav(source, wav_path)
        
        print("-> Analyzing acoustic features (RMS, Vocal presence, Pitch stability)... (This might take a moment)")
        segments = detect_segments(
            wav_path=wav_path,
            threshold=threshold,
            min_duration=min_duration,
            merge_gap=merge_gap,
            min_loudness_lift=min_loudness_lift,
        )
        
        # Print results in a markdown table
        lines = [
            f"### 🎵 Analysis complete for Bilibili Video ID: `{video_id}`",
            f"Found **{len(segments)}** singing segment(s) matching your parameters.",
            "",
            "| Start | End | Duration | Confidence | Loudness Lift | Voice Presence | Pitch Stability |",
            "|---:|---:|---:|---:|---:|---:|---:|",
        ]
        for seg in segments:
            lines.append(
                f"| **{fmt_time(seg.start)}** | **{fmt_time(seg.end)}** | {fmt_time(seg.duration)} | {seg.confidence:.2f} | {seg.loudness_lift:.2f} | {seg.voice_presence:.2f} | {seg.pitch_stability:.2f} |"
            )
        if not segments:
            lines.append(
                "| No segments matched | - | - | - | - | - | - |"
            )
        
        display(Markdown("\n".join(lines)))
else:
    print("Could not process because the audio file was not found.")